# Prompt Preview

Ce notebook lit `data/raw/swissdox_input.parquet`, formate chaque article dans le template de prompt défini dans `src/prompts.py`, et produit un CSV avec :
- **`prompt_ready`** : le prompt complet (system + user) à coller dans un chat LLM en ligne
- **`llm_answer`** : colonne vide à remplir manuellement avec la réponse du LLM (8 lignes attendues)

La colonne `text` brute est exclue pour alléger le fichier.

### Format de réponse attendu du LLM
```
SWISS_CONTEXT: YES or NO
CRITICISM: YES or NO or N/A
TARGETED_ENTITY_TYPE: [Federal Department | Federal Agency or Civil Servant | Regulatory Agency | Political Figure | Political Party | Canton | Municipality | Other] or N/A
TARGETED_ENTITY_NAME: [name as in text] or N/A
SOURCE_TYPE: [Journalist | Lobby or Private Interest Group | Another Federal Department | Politician or Party | General Public or Civil Society | Foreign Entity | Other] or N/A
SOURCE_NAME: [name as in text] or N/A
CRITICISM_TOPIC: [1-2 sentences] or N/A
POPULIST_RHETORIC: YES or NO or N/A
```

In [ ]:
import sys
import importlib
import pandas as pd
from pathlib import Path

# Ajouter src/ au path pour importer prompts.py
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import prompts
importlib.reload(prompts)  # force le rechargement si le kernel tourne déjà
from prompts import build_user_prompt, SYSTEM_PROMPT

In [ ]:
# Charger les données
df = pd.read_parquet('../data/raw/swissdox_input.parquet')
print(f"Articles chargés : {len(df):,}")
df.head(2)

In [ ]:
# Construire la colonne prompt_ready (system prompt + user prompt fusionnés)
df['prompt_ready'] = df.apply(
    lambda row: SYSTEM_PROMPT + "\n\n" + build_user_prompt(row, 'text'), axis=1
)

# Ajouter la colonne vide pour les réponses LLM
df['llm_answer'] = ''

print(f"Exemple de prompt_ready pour le premier article :")
print("-" * 60)
print(df['prompt_ready'].iloc[0])

In [ ]:
# Supprimer la colonne text brute et exporter
cols_to_keep = [c for c in df.columns if c != 'text']
df_out = df[cols_to_keep]

out_path = Path('../data/raw/swissdox_prompts.csv')
df_out.to_csv(out_path, index=False)

print(f"Fichier exporté : {out_path}")
print(f"Colonnes : {df_out.columns.tolist()}")
print(f"Lignes : {len(df_out):,}")

In [ ]:
# Aperçu du résultat final
df_out[['article_id', 'medium_name', 'pubtime', 'prompt_ready', 'llm_answer']].head(3)